In [ ]:
"""
Week 1 - Day 5
Final Wrap Up
==============
Complete Week 1 summary demonstrating
all deliverables for mid review.

Week 1 Achievements:
→ Custom Gym Environment built
→ 5 Baseline agents implemented
→ Q-Learning agent trained
→ Q-Table policy analyzed

Infotact DS/ML Internship — Project 2
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
import sys

CURRENT_DIR = os.getcwd()

# If running from project root
if os.path.exists(os.path.join(CURRENT_DIR, "src")):
    SRC_DIR = os.path.join(CURRENT_DIR, "src")
else:
    # If running from notebooks/week1
    SRC_DIR = os.path.abspath(os.path.join(CURRENT_DIR, "..", "..", "src"))

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("SRC Path:", SRC_DIR)

from environment.pricing_env import (
    DynamicPricingEnv,
    PRICE_LEVELS
)
from agents.baseline_agents import (
    FixedPriceAgent,
    RandomAgent,
    TimedPricingAgent,
    DemandBasedAgent,
    LinearDecayAgent
)
from agents.q_learning_agent import (
    QLearningAgent,
    QL_CONFIG
)
from utils.evaluator import (
    evaluate_agent,
    compare_agents
)

plt.style.use('seaborn-v0_8')
print("✅ Week 1 Final Wrap modules loaded!")

In [ ]:
env = DynamicPricingEnv()
obs, _ = env.reset(seed=42)

print("╔══════════════════════════════════════════╗")
print("║   DYNAMIC PRICING ENVIRONMENT SPECS      ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Max Inventory  : {env.max_inventory}"
      f"{'':<23} ║")
print(f"║  Max Days       : {env.max_days}"
      f"{'':<23} ║")
print(f"║  Price Levels   : {env.price_levels}"
      f"{'':<6} ║")
print(f"║  Action Space   : {env.action_space}"
      f"{'':<17} ║")
print(f"║  State Space    : "
      f"{(env.max_inventory+1)*(env.max_days+1)}"
      f" states{'':<13} ║")
print("╠══════════════════════════════════════════╣")
print("║  MDP FORMULATION:                        ║")
print("║  S = (inventory, days_left)              ║")
print("║  A = price_level ($50 to $300)           ║")
print("║  R = revenue earned per sale             ║")
print("║  P = -10 per unsold ticket               ║")
print("╚══════════════════════════════════════════╝")

In [ ]:
print("Training and evaluating all agents...")
print("Please wait ~3 minutes\n")

# Create all agents
all_agents = [
    FixedPriceAgent(env, price_idx=2),
    RandomAgent(env),
    TimedPricingAgent(env),
    DemandBasedAgent(env),
    LinearDecayAgent(env),
]

# Run baseline comparison
all_results, summary_df = compare_agents(
    all_agents,
    n_episodes=100
)

# Train Q-Learning
print("\nTraining Q-Learning agent...")
ql_agent = QLearningAgent(env, QL_CONFIG)
ql_rewards = ql_agent.train(
    n_episodes=5000,
    verbose=False
)
ql_eval = ql_agent.evaluate(n_episodes=100)
print(f"✅ Q-Learning: ${ql_eval['mean_revenue']:.0f}")

In [ ]:
# Add Q-Learning to comparison
comparison = {}
for _, row in summary_df.iterrows():
    comparison[row['Agent']] = row['Mean Revenue']
comparison['Q-Learning 🤖'] = ql_eval['mean_revenue']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Bar chart
names    = list(comparison.keys())
revenues = list(comparison.values())
colors   = [
    'gold' if '🤖' in n
    else 'steelblue'
    for n in names
]

bars = axes[0].bar(
    names, revenues,
    color=colors,
    edgecolor='black',
    width=0.6
)
axes[0].set_title(
    'All Agents — Mean Revenue\n'
    'Q-Learning vs Baselines',
    fontweight='bold',
    fontsize=12
)
axes[0].set_ylabel('Mean Revenue ($)')
axes[0].set_xticklabels(
    names, rotation=20, fontsize=9
)
for bar, val in zip(bars, revenues):
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        val + 30,
        f'${val:.0f}',
        ha='center',
        fontsize=9,
        fontweight='bold'
    )

# Q-Learning training curve
smooth = pd.Series(ql_rewards).rolling(
    window=100
).mean()
axes[1].plot(
    ql_rewards,
    alpha=0.2,
    color='steelblue',
    linewidth=0.5
)
axes[1].plot(
    smooth,
    color='red',
    linewidth=2.5,
    label='100-ep Average'
)
axes[1].set_title(
    'Q-Learning Training Curve\n'
    'Revenue Improving Over Time',
    fontweight='bold',
    fontsize=12
)
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Revenue ($)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(
    'Week 1 Final Results — RL Dynamic Pricing',
    fontsize=14,
    fontweight='bold'
)
import os

RESULTS_DIR = os.path.join(os.getcwd(), "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

SAVE_PATH = os.path.join(
    RESULTS_DIR,
    "week1_final_comparison.png"
)

plt.tight_layout()

plt.savefig(
    SAVE_PATH,
    dpi=150,
    bbox_inches="tight"
)

print(f"✅ Saved: {SAVE_PATH}")

plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

agent_list = [
    (FixedPriceAgent(env, 2), 'steelblue'),
    (TimedPricingAgent(env),  'coral'),
    (DemandBasedAgent(env),   'green'),
    (LinearDecayAgent(env),   'purple'),
    (ql_agent,                'gold'),
]

for i, (agent, color) in enumerate(agent_list):
    if hasattr(agent, 'run_episode'):
        result = agent.run_episode(seed=42)
    else:
        # Q-Learning episode
        state, _ = env.reset(seed=42)
        prices_used = []
        total_rev   = 0
        done        = False
        while not done:
            action = ql_agent.select_action(
                state, training=False
            )
            state, reward, term, trunc, info = (
                env.step(action)
            )
            prices_used.append(info['price'])
            total_rev  += max(0, reward)
            done = term or trunc
        result = {
            'prices_used'   : prices_used,
            'total_revenue' : total_rev
        }

    prices = result['prices_used']
    name   = getattr(agent, 'name', 'Q-Learning')

    axes[i].plot(
        prices,
        color=color,
        linewidth=2,
        marker='o',
        markersize=3
    )
    axes[i].fill_between(
        range(len(prices)),
        prices,
        alpha=0.15,
        color=color
    )
    axes[i].set_title(
        f'{name}\n'
        f"Revenue: ${result['total_revenue']:.0f}",
        fontweight='bold'
    )
    axes[i].set_xlabel('Day')
    axes[i].set_ylabel('Price ($)')
    axes[i].set_ylim([0, 350])
    axes[i].grid(True, alpha=0.3)

axes[-1].axis('off')
plt.suptitle(
    'Price Trajectories — All Agents\n'
    'How Each Agent Prices Over 30 Days',
    fontsize=13,
    fontweight='bold'
)
import os

RESULTS_DIR = os.path.join(os.getcwd(), "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

SAVE_PATH = os.path.join(
    RESULTS_DIR,
    "week1_trajectories.png"
)

plt.tight_layout()

plt.savefig(
    SAVE_PATH,
    dpi=150,
    bbox_inches="tight"
)

print(f"✅ Saved: {SAVE_PATH}")

plt.show()
print("✅ Trajectory comparison saved!")

In [ ]:
prices_policy = ql_agent.get_price_policy()

plt.figure(figsize=(14, 7))
sns.heatmap(
    prices_policy,
    cmap='RdYlGn',
    cbar_kws={'label': 'Optimal Price ($)'},
    xticklabels=5,
    yticklabels=5
)
plt.title(
    'Learned Q-Table Policy\n'
    'Optimal Price by (Inventory, Days Left)',
    fontweight='bold',
    fontsize=13
)
plt.xlabel('Days Until Departure')
plt.ylabel('Remaining Inventory')
plt.tight_layout()
import os

RESULTS_DIR = os.path.join(os.getcwd(), "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

SAVE_PATH = os.path.join(
    RESULTS_DIR,
    "week1_q_policy.png"
)

plt.tight_layout()

plt.savefig(
    SAVE_PATH,
    dpi=150,
    bbox_inches="tight"
)

print(f"✅ Saved: {SAVE_PATH}")
plt.show()
print("✅ Q-Table policy saved!")

In [ ]:
best_baseline = max(
    v for k, v in comparison.items()
    if '🤖' not in k
)
ql_rev = comparison['Q-Learning 🤖']
improvement = (ql_rev - best_baseline) / \
               best_baseline * 100

print("╔══════════════════════════════════════════╗")
print("║         WEEK 1 FINAL SUMMARY             ║")
print("╠══════════════════════════════════════════╣")
print("║  DELIVERABLES:                           ║")
print("║  ✅ MDP formulated                       ║")
print("║  ✅ Custom Gym Environment built         ║")
print("║  ✅ Stochastic demand function           ║")
print("║  ✅ 5 Baseline agents implemented        ║")
print("║  ✅ Q-Learning agent trained             ║")
print("║  ✅ Q-Table policy analyzed              ║")
print("╠══════════════════════════════════════════╣")
print("║  RESULTS:                                ║")
print(f"║  Best Baseline  : ${best_baseline:.0f}"
      f"{'':<22} ║")
print(f"║  Q-Learning     : ${ql_rev:.0f}"
      f"{'':<22} ║")
print(f"║  Improvement    : {improvement:+.1f}%"
      f"{'':<23} ║")
print("╠══════════════════════════════════════════╣")
print("║  GITHUB:                                 ║")
print("║  Issues #1 #2 #3 #4 → CLOSED ✅         ║")
print("║  Issue #5 DQN → IN PROGRESS 🔄          ║")
print("╠══════════════════════════════════════════╣")
print("║  WEEK 2 PLAN:                            ║")
print("║  → Deep Q-Network (DQN)                  ║")
print("║  → Neural Network replaces Q-table       ║")
print("║  → Experience replay buffer              ║")
print("║  → Epsilon greedy strategy               ║")
print("╠══════════════════════════════════════════╣")
print("║  🎯 TARGET: DQN beats all baselines!    ║")
print("╚══════════════════════════════════════════╝")